# 🏥 NFL Injury Ingestion - Historical & Current

## 🎯 Purpose
Track player injuries across multiple seasons with weekly granularity for fantasy football analysis.

## 📊 Data Sources

### 1. **nflverse Historical Injuries** (2016-2026)
* Weekly injury reports for 11 years
* Official NFL injury designations
* Comprehensive coverage: Out, Questionable, Doubtful, IR, PUP
* Source: `nfl.import_injuries()`

### 2. **Sleeper Current Injuries** (2026+)
* Real-time injury updates
* Current roster status
* Player depth chart impact
* Source: Sleeper API

## 🔄 Operating Modes

**HISTORICAL_MODE = True**
* One-time backfill of 2016-2026 injury data
* ~55,000+ injury records across 11 seasons
* Run once to establish baseline

**HISTORICAL_MODE = False**
* Weekly updates for current season only
* Fetches latest week's injuries
* Append to existing historical data
* Run weekly during NFL season

## 📅 Season/Week Logic

**Automatic season detection:**
* Current date determines season year
* Week calculation:
  * Before August 1: Offseason (week = '1')
  * August 1-15: preseason1
  * August 16-23: preseason2
  * August 24-31: preseason3
  * September+: Regular season weeks 1-18

## 💾 Output Table

`main.fantasai.silver_injury_reports_historical`

Schema:
* season, week, player_id, player_name, position, team
* injury_status, injury_body_part, injury_notes
* source (nflverse_historical / sleeper_current)
* fetched_at

In [0]:
from datetime import datetime, timedelta
import calendar

def get_current_season_week():
    """
    Determine current NFL season and week based on today's date.
    
    Returns:
        tuple: (season_year, week_string)
        
    Week progression:
    - Before August 1: Offseason, week = '1'
    - August 1-15: 'preseason1'
    - August 16-23: 'preseason2'
    - August 24-31: 'preseason3'
    - September - February: Regular season weeks '1' through '18'
    """
    today = datetime.now()
    year = today.year
    month = today.month
    day = today.day
    
    # Determine season year
    # NFL season spans Sept (year N) to Feb (year N+1)
    # So Jan-July belongs to previous season, Aug-Dec to current season
    if month >= 1 and month <= 7:
        # Jan-July: Previous season still active or offseason
        season = year
    else:
        # Aug-Dec: New season starting
        season = year
    
    # Determine week
    if month < 8:  # Jan-July: Offseason
        week = '1'
    elif month == 8:  # August: Preseason
        if day <= 15:
            week = 'preseason1'
        elif day <= 23:
            week = 'preseason2'
        else:
            week = 'preseason3'
    else:  # Sept+: Regular season
        # Approximate week calculation (first Sunday of Sept = Week 1)
        # Find first Sunday of September
        first_day = datetime(year, 9, 1)
        days_until_sunday = (6 - first_day.weekday()) % 7
        first_sunday = first_day + timedelta(days=days_until_sunday)
        
        # Calculate weeks since first Sunday
        days_since_start = (today - first_sunday).days
        week_num = max(1, min(18, (days_since_start // 7) + 1))
        week = str(week_num)
    
    return season, week

# Test the function
current_season, current_week = get_current_season_week()
print(f"📅 Current Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"🏈 NFL Season: {current_season}")
print(f"📆 NFL Week: {current_week}")

In [0]:
# =============================================================================
# CONFIGURATION MODE
# =============================================================================
# Set to True to fetch ALL historical injuries (2016-2026)
# Set to False to fetch only CURRENT WEEK (for weekly updates)
HISTORICAL_MODE = False  # ✅ Historical backfill complete - now in weekly mode
# =============================================================================

# Historical range (nflverse injury data availability)
HISTORICAL_START_SEASON = 2016
HISTORICAL_END_SEASON = 2026

# Current season and week (auto-detected)
CURRENT_SEASON, CURRENT_WEEK = get_current_season_week()

if HISTORICAL_MODE:
    print("🔄 HISTORICAL MODE: Will fetch ALL injury data 2016-2026")
    print(f"   Seasons: {HISTORICAL_START_SEASON}-{HISTORICAL_END_SEASON}")
    print(f"   Source: nflverse injury reports")
    print(f"   Expected: ~55,000+ injury records")
    print(f"\n⚠️  Run this ONCE to establish baseline")
    print(f"   Then switch to HISTORICAL_MODE = False for weekly updates")
else:
    print("⚡ LATEST MODE: Will fetch current week only")
    print(f"   Season: {CURRENT_SEASON}")
    print(f"   Week: {CURRENT_WEEK}")
    print(f"   Source: Sleeper API (current injuries)")
    print(f"   Expected: ~100-200 injury records")
    print("\n📌 Run this weekly during NFL season for updates")

print("\n📊 Data Sources:")
print("   - nflverse: Historical injuries (2016-2026)")
print("   - Sleeper API: Current injuries (2026+)")
print("\n💾 Output Table:")
print("   - main.fantasai.silver_injury_reports_historical")

In [0]:
%pip install nfl_data_py requests --quiet

import nfl_data_py as nfl
import requests
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

print("✅ Dependencies installed and imported")

In [0]:
# Fetch historical injury data from nflverse

historical_injuries_df = pd.DataFrame()

if HISTORICAL_MODE:
    print(f"\n{'='*80}")
    print(f"📅 NFLVERSE HISTORICAL INJURIES INGESTION")
    print(f"{'='*80}\n")
    
    print(f"📅 Fetching ALL injury data from {HISTORICAL_START_SEASON} to {HISTORICAL_END_SEASON}...\n")
    
    # Fetch all seasons at once
    seasons_to_fetch = list(range(HISTORICAL_START_SEASON, HISTORICAL_END_SEASON + 1))
    
    print(f"   Seasons: {seasons_to_fetch}")
    print(f"   Expected: ~5,000 injury records per season")
    print(f"   Total expected: ~{len(seasons_to_fetch) * 5000:,} records\n")
    
    try:
        historical_injuries_df = nfl.import_injuries(seasons_to_fetch)
        
        if historical_injuries_df is not None and len(historical_injuries_df) > 0:
            print(f"✅ Fetched {len(historical_injuries_df):,} injury records")
            print(f"   Seasons: {historical_injuries_df['season'].min()}-{historical_injuries_df['season'].max()}")
            print(f"   Weeks: {historical_injuries_df['week'].min()}-{historical_injuries_df['week'].max()}")
            print(f"   Unique players: {historical_injuries_df['full_name'].nunique():,}")
            
            # Standardize column names to match our schema
            historical_injuries_df = historical_injuries_df.rename(columns={
                'full_name': 'player_name',
                'report_status': 'injury_status',
                'report_primary_injury': 'injury_body_part',
                'report_secondary_injury': 'injury_notes',
                'gsis_id': 'player_id'
            })
            
            # Add source column
            historical_injuries_df['source'] = 'nflverse_historical'
            
            # Convert week to string for consistency
            historical_injuries_df['week'] = historical_injuries_df['week'].astype(str)
            
            print(f"\n🏥 Injury Status Breakdown:")
            print(historical_injuries_df['injury_status'].value_counts().head(10))
            
            print(f"\n📊 Sample injury data:")
            display(historical_injuries_df[['season', 'week', 'team', 'player_name', 'position', 'injury_body_part', 'injury_status']].head(20))
        else:
            print("⚠️ No historical injury data returned")
            
    except Exception as e:
        print(f"❌ Error fetching nflverse injuries: {e}")
        print(f"   Type: {type(e).__name__}")
else:
    print("\n⏭️ Skipping historical injuries fetch (HISTORICAL_MODE = False)")
    print("   Historical data should already be loaded")
    print("   Only fetching current week from Sleeper...")

In [0]:
# Fetch current injuries from Sleeper API

current_injuries_df = pd.DataFrame()

if not HISTORICAL_MODE:
    print(f"\n{'='*80}")
    print(f"🏥 SLEEPER CURRENT INJURIES INGESTION")
    print(f"{'='*80}\n")
    
    print(f"📅 Fetching current injuries for Season {CURRENT_SEASON}, Week {CURRENT_WEEK}...\n")
    
    try:
        # Fetch all NFL players from Sleeper
        url = "https://api.sleeper.app/v1/players/nfl"
        print(f"📁 Calling: {url}")
        
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        players_data = response.json()
        
        print(f"✅ Fetched {len(players_data):,} players")
        
        # Extract relevant fantasy positions
        relevant_positions = ['QB', 'RB', 'WR', 'TE', 'K', 'DEF']
        
        # Parse player data
        players_list = []
        for player_id, player in players_data.items():
            if player.get('position') in relevant_positions:
                players_list.append({
                    'player_id': player_id,
                    'player_name': player.get('full_name') or f"{player.get('first_name', '')} {player.get('last_name', '')}".strip(),
                    'position': player.get('position'),
                    'team': player.get('team'),
                    'injury_status': player.get('injury_status'),
                    'injury_body_part': player.get('injury_body_part'),
                    'injury_notes': player.get('injury_notes'),
                    'injury_start_date': player.get('injury_start_date')
                })
        
        all_players_df = pd.DataFrame(players_list)
        
        # Filter to only players with active injuries
        current_injuries_df = all_players_df[
            all_players_df['injury_status'].notna()
        ].copy()
        
        if len(current_injuries_df) > 0:
            # Add season and week
            current_injuries_df['season'] = CURRENT_SEASON
            current_injuries_df['week'] = CURRENT_WEEK
            current_injuries_df['source'] = 'sleeper_current'
            
            print(f"✅ Extracted {len(current_injuries_df)} current injuries")
            print(f"\n🏥 Injury Status Breakdown:")
            print(current_injuries_df['injury_status'].value_counts())
            
            print(f"\n📊 Sample current injuries:")
            display(current_injuries_df[['season', 'week', 'team', 'player_name', 'position', 'injury_body_part', 'injury_status']].head(20))
        else:
            print("⚠️ No current injuries found")
            
    except Exception as e:
        print(f"❌ Error fetching Sleeper injuries: {e}")
        print(f"   Type: {type(e).__name__}")
else:
    print("\n⏭️ Skipping Sleeper current injuries fetch (HISTORICAL_MODE = True)")
    print("   Focus is on historical backfill")
    print("   Current injuries will be added in weekly update mode")

In [0]:
# Combine historical and current injuries into single DataFrame

print(f"\n{'='*80}")
print(f"🔀 MERGING INJURY DATA")
print(f"{'='*80}\n")

all_injuries_df = pd.DataFrame()

if HISTORICAL_MODE and len(historical_injuries_df) > 0:
    all_injuries_df = historical_injuries_df
    print(f"📅 Historical injuries: {len(historical_injuries_df):,} records")
elif not HISTORICAL_MODE and len(current_injuries_df) > 0:
    all_injuries_df = current_injuries_df
    print(f"📅 Current injuries: {len(current_injuries_df):,} records")
else:
    print("⚠️ No injury data to merge")

if len(all_injuries_df) > 0:
    # Ensure all required columns exist
    required_columns = [
        'season', 'week', 'player_id', 'player_name', 'position', 'team',
        'injury_status', 'injury_body_part', 'injury_notes', 'source'
    ]
    
    for col in required_columns:
        if col not in all_injuries_df.columns:
            all_injuries_df[col] = None
    
    # Select only required columns
    all_injuries_df = all_injuries_df[required_columns]
    
    # Convert to Spark DataFrame
    print(f"\n🔄 Converting to Spark DataFrame...")
    injuries_spark_df = spark.createDataFrame(all_injuries_df)
    
    # Add fetched_at timestamp
    injuries_spark_df = injuries_spark_df.withColumn("fetched_at", F.current_timestamp())
    
    print(f"✅ Spark DataFrame created: {injuries_spark_df.count():,} records")
    print(f"\n📊 Sample merged data:")
    display(injuries_spark_df.select('season', 'week', 'team', 'player_name', 'position', 'injury_status', 'source').limit(20))
else:
    print("❌ No data to process")

In [0]:
# Write injury data to Delta table

if len(all_injuries_df) > 0:
    print(f"\n{'='*80}")
    print(f"💾 WRITING TO DELTA TABLE")
    print(f"{'='*80}\n")
    
    table_name = "main.fantasai.silver_injury_reports_historical"
    
    # Create table if it doesn't exist
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        season INT,
        week STRING,
        player_id STRING,
        player_name STRING,
        position STRING,
        team STRING,
        injury_status STRING,
        injury_body_part STRING,
        injury_notes STRING,
        source STRING,
        fetched_at TIMESTAMP
    )
    USING DELTA
    COMMENT 'Historical and current NFL injury reports with weekly tracking'
    """)
    print(f"✅ Table created/verified: {table_name}")
    
    # Create temp view for merge
    injuries_spark_df.createOrReplaceTempView("injuries_staging")
    
    # Get record count before merge
    records_to_merge = injuries_spark_df.count()
    
    # Merge into table (upsert based on season, week, player_id)
    print(f"\n🔄 Merging data into {table_name}...")
    
    merge_query = f"""
    MERGE INTO {table_name} AS target
    USING injuries_staging AS source
    ON target.season = source.season 
       AND target.week = source.week 
       AND target.player_id = source.player_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """
    
    spark.sql(merge_query)
    
    # Get final count
    final_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]['cnt']
    
    if HISTORICAL_MODE:
        print(f"✅ Historical backfill complete!")
        print(f"   Added/Updated: {records_to_merge:,} injury records")
        print(f"   Total in table: {final_count:,} records")
        print(f"\n⚠️  NEXT STEP: Set HISTORICAL_MODE = False for weekly updates")
    else:
        print(f"✅ Weekly update complete!")
        print(f"   Added/Updated: {records_to_merge:,} injury records (Week {CURRENT_WEEK})")
        print(f"   Total in table: {final_count:,} records")
        print(f"\n📅 Run this notebook weekly to track injury changes")
else:
    print("\n⏭️ Skipping write - no data to save")

In [0]:
%sql
-- Verify injury data coverage by season
SELECT 
  season,
  COUNT(DISTINCT week) as weeks_with_data,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_injury_records,
  COUNT(DISTINCT injury_status) as status_types,
  source
FROM main.fantasai.silver_injury_reports_historical
GROUP BY season, source
ORDER BY season DESC, source

In [0]:
%sql
-- Injury status breakdown for recent seasons
SELECT 
  season,
  injury_status,
  COUNT(*) as injury_count,
  COUNT(DISTINCT player_id) as unique_players
FROM main.fantasai.silver_injury_reports_historical
WHERE season >= 2021
GROUP BY season, injury_status
ORDER BY season DESC, injury_count DESC

In [0]:
%sql
-- Show most recent week's injuries
WITH latest_week AS (
  SELECT MAX(season) as max_season
  FROM main.fantasai.silver_injury_reports_historical
),
latest_season_week AS (
  SELECT season, MAX(CAST(week AS INT)) as max_week
  FROM main.fantasai.silver_injury_reports_historical
  WHERE season = (SELECT max_season FROM latest_week)
    AND week NOT LIKE 'preseason%'
  GROUP BY season
)
SELECT 
  i.season,
  i.week,
  i.player_name,
  i.position,
  i.team,
  i.injury_status,
  i.injury_body_part,
  i.injury_notes,
  i.source,
  i.fetched_at
FROM main.fantasai.silver_injury_reports_historical i
JOIN latest_season_week l 
  ON i.season = l.season 
  AND CAST(i.week AS INT) = l.max_week
WHERE i.injury_status IS NOT NULL
ORDER BY 
  CASE i.injury_status
    WHEN 'Out' THEN 1
    WHEN 'Doubtful' THEN 2
    WHEN 'Questionable' THEN 3
    WHEN 'IR' THEN 4
    ELSE 5
  END,
  i.player_name
LIMIT 50

## 📝 Usage Instructions

### 🎯 First Time Setup (Historical Backfill)

1. **Set Configuration:**
   ```python
   HISTORICAL_MODE = True
   ```

2. **Run All Cells:**
   * Fetches 10 years of injury data (2016-2025)
   * ~50,000+ injury records from nflverse
   * Populates `main.fantasai.silver_injury_reports_historical`

3. **Verify Coverage:**
   * Check "Verify Historical Injury Coverage" cell
   * Confirm all seasons 2016-2025 are present

### 🔄 Weekly Updates (During Season)

1. **Switch to Latest Mode:**
   ```python
   HISTORICAL_MODE = False
   ```

2. **Run Weekly:**
   * Fetches current week injuries from Sleeper API
   * Adds ~100-200 injury records per week
   * Appends to existing historical data

3. **Recommended Schedule:**
   * **Tuesday Evening**: After Monday Night Football
   * **Friday Morning**: Before weekend lineup decisions
   * **Sunday Morning**: Final injury updates before games

### 📊 Data Coverage

**Historical (2016-2025):**
* Source: nflverse `import_injuries()`
* Weekly injury reports for all NFL players
* Official designations: Out, Questionable, Doubtful, IR, PUP

**Current (2026+):**
* Source: Sleeper API
* Real-time injury status
* Depth chart impact
* Fantasy-relevant players only

### 💾 Output Table

`main.fantasai.silver_injury_reports_historical`

**Schema:**
* `season` (INT) - NFL season year
* `week` (STRING) - preseason1-3, 1-18
* `player_id`, `player_name`, `position`, `team`
* `injury_status` - Out, Questionable, Doubtful, IR, etc.
* `injury_body_part` - Knee, Ankle, Shoulder, etc.
* `injury_notes` - Additional details
* `source` - nflverse_historical / sleeper_current
* `fetched_at` - Timestamp

### ⚠️ Important Notes

1. **One-Time Historical Load:**
   * Only run HISTORICAL_MODE = True ONCE
   * After initial backfill, switch to False
   * Historical data doesn't change, no need to re-fetch

2. **Week Detection:**
   * Automatically determines current season/week
   * Handles offseason, preseason, regular season
   * No manual configuration needed

3. **Data Quality:**
   * nflverse: Official NFL injury reports (most reliable)
   * Sleeper: Community-sourced, updated frequently
   * Both sources complement each other

### 🔗 Related Tables

* `main.fantasai.silver_weekly_stats` - Player performance data
* `main.fantasai.player_snap_counts` - Snap count tracking
* `main.fantasai.bronze_nfl_weather` - Weather conditions
* `main.fantasai.silver_player_news` - Player news updates

**Combine for Analysis:**
```sql
SELECT 
  s.player_name,
  s.position,
  s.fantasy_points,
  i.injury_status,
  i.injury_body_part,
  w.wind_speed_mph,
  n.news_updated
FROM main.fantasai.silver_weekly_stats s
LEFT JOIN main.fantasai.silver_injury_reports_historical i
  ON s.player_id = i.player_id 
  AND s.season = i.season 
  AND s.week = i.week
LEFT JOIN main.fantasai.bronze_nfl_weather w
  ON s.season = w.season 
  AND s.week = w.week
LEFT JOIN main.fantasai.silver_player_news n
  ON s.player_id = n.player_id
WHERE s.season = 2024 AND s.week = '5'
```